In [1]:
# ============================================
# YOLO MODEL OPTIMIZATION
# Environment Setup
# ============================================

!nvidia-smi

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Sat Sep 26 13:59:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================
# INSTALL OPTIMIZATION LIBRARIES
# ============================================

!pip install -q ultralytics==8.4.27 torch-pruning==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.2 MB/s eta 0:00:00


In [4]:
# ============================================
# VERIFY OPTIMIZATION ENVIRONMENT
# ============================================

import torch
import ultralytics
import torch_pruning as tp
import importlib.metadata

print("========== ENVIRONMENT ==========")
print("PyTorch       :", torch.__version__)
print("Ultralytics   :", ultralytics.__version__)
print("Torch-Pruning :", importlib.metadata.version("torch-pruning"))
print("CUDA          :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("CUDA version  :", torch.version.cuda)

print("=================================")

========== ENVIRONMENT ==========
PyTorch       : 2.11.0+cu128
Ultralytics   : 8.4.27
Torch-Pruning : 1.6.0
CUDA          : True
GPU           : Tesla T4
CUDA version  : 12.8


In [5]:
# ============================================
# UPLOAD ORIGINAL BASELINE MODEL
# ============================================

from google.colab import files
import os

print("Select your ORIGINAL best.pt file")
uploaded = files.upload()

uploaded_name = list(uploaded.keys())[0]

# Keep a clear, fixed name inside Colab
baseline_model_path = "/content/baseline_best.pt"

os.rename(f"/content/{uploaded_name}", baseline_model_path)

print("\nModel uploaded successfully!")
print("Path:", baseline_model_path)
print("Size:", os.path.getsize(baseline_model_path) / (1024 * 1024), "MB")

Select your ORIGINAL best.pt file


Saving best.pt to best.pt

Model uploaded successfully!
Path: /content/baseline_best.pt
Size: 49.606889724731445 MB


In [6]:
# ============================================
# LOAD BASELINE MODEL
# ============================================

from ultralytics import YOLO
import torch

MODEL_PATH = "/content/baseline_best.pt"

model = YOLO(MODEL_PATH)

print("========== BASELINE MODEL ==========")
print("Model loaded successfully")
print("Device:", "CUDA (T4)" if torch.cuda.is_available() else "CPU")

print("\nModel architecture:")
print(model.model)

print("\n====================================")

========== BASELINE MODEL ==========
Model loaded successfully
Device: CUDA (T4)

Model architecture:
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.00